In [57]:
import pandas as pd

In [58]:
df = pd.read_csv("dados/confrontos/historico_confrontos.csv")
display(df.head())

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [59]:
df = df.drop(columns=["tournament","city","country","neutral"])

In [60]:
from datetime import date

def convert_data_string(data_string: str):
    year, month, day = list(map(int, data_string.split("-")))
    return date(year,month,day)

df["date"] = df["date"].apply(convert_data_string)

display(df.head())

,date,home_team,away_team,home_score,away_score
0,1872-11-30,Scotland,England,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0
2,1874-03-07,Scotland,England,2.0,1.0
3,1875-03-06,England,Scotland,2.0,2.0
4,1876-03-04,Scotland,England,3.0,0.0


In [61]:
fim_copa_22 = date(year=2022,month=12, day=18)
inicio_copa26 = date(year=2026,month=6,day=11)
df_ciclo26 = df[(df["date"] > fim_copa_22) & (df["date"] < inicio_copa26)]

display(df_ciclo26)

,date,home_team,away_team,home_score,away_score
45787,2022-12-20,Cambodia,Philippines,3.0,2.0
45788,2022-12-20,Brunei,Thailand,0.0,5.0
45789,2022-12-21,Myanmar,Malaysia,0.0,1.0
45790,2022-12-21,Laos,Vietnam,0.0,6.0
45791,2022-12-23,Oman,Syria,2.0,1.0
...,...,...,...,...,...
49400,2026-06-09,Iraq,Venezuela,0.0,2.0
49401,2026-06-10,Bolivia,Algeria,0.0,4.0
49402,2026-06-10,England,Costa Rica,3.0,0.0
49403,2026-06-10,Portugal,Nigeria,2.0,1.0


In [62]:
qualified_teams = [
    "Canada",
    "United States",
    "Mexico",
    "Curacao",
    "Haiti",
    "Panama",
    "Japan",
    "Iran",
    "Uzbekistan",
    "South Korea",
    "Jordan",
    "Australia",
    "Qatar",
    "Saudi Arabia",
    "New Zealand",
    "Argentina",
    "Brazil",
    "Ecuador",
    "Uruguay",
    "Colombia",
    "Paraguay",
    "Morocco",
    "Tunisia",
    "Egypt",
    "Algeria",
    "Ghana",
    "Cape Verde",
    "South Africa",
    "Ivory Coast",
    "Senegal",
    "England",
    "France",
    "Croatia",
    "Portugal",
    "Norway",
    "Netherlands",
    "Germany",
    "Switzerland",
    "Austria",
    "Belgium",
    "Spain",
    "Scotland",
    "Turkey",
    "Czech Republic",
    "Sweden",
    "Bosnia and Herzegovina",
    "DR Congo",
    "Iraq"
]

df_ciclo26 = df_ciclo26[(df["home_team"].isin(qualified_teams)) | (df["away_score"].isin(qualified_teams))]
df_ciclo26 = df_ciclo26.reset_index()

/tmp/ipykernel_20762/3298589592.py:52: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_ciclo26 = df_ciclo26[(df["home_team"].isin(qualified_teams)) | (df["away_score"].isin(qualified_teams))]


In [63]:
print(df_ciclo26.shape)

(1091, 6)


In [64]:
display(df_ciclo26)

,index,date,home_team,away_team,home_score,away_score
0,45802,2022-12-30,Iraq,Kuwait,1.0,0.0
1,45811,2023-01-06,Iraq,Oman,0.0,0.0
2,45818,2023-01-09,Sweden,Finland,2.0,0.0
3,45820,2023-01-09,Iraq,Saudi Arabia,2.0,0.0
4,45823,2023-01-10,Qatar,Bahrain,1.0,2.0
...,...,...,...,...,...,...
1086,49386,2026-06-09,DR Congo,Chile,1.0,2.0
1087,49392,2026-06-09,Saudi Arabia,Senegal,0.0,0.0
1088,49400,2026-06-09,Iraq,Venezuela,0.0,2.0
1089,49402,2026-06-10,England,Costa Rica,3.0,0.0


In [65]:
df_ciclo26.to_csv("dados/confrontos/historico_confrontos_ciclo2026.csv")

anos_copa = [i for i in range(1930,2023,4)]

In [66]:
def stats_cicle(ano):
    
    df_confrontos = pd.read_csv(f"dados/confrontos/historico_confrontos_ciclo{ano}.csv")

    qualified_teams = pd.read_csv(f"dados/copas/FIFA - {ano}.csv")
    qualified_teams = list(qualified_teams["Team"])
    print(qualified_teams)

    df_stats = pd.DataFrame({"team": qualified_teams})
    df_stats = df_stats.set_index("team")
    df_stats["wins"] = 0
    df_stats["draws"] = 0
    df_stats["loses"] = 0
    df_stats["points"] = 0
    df_stats["games_played"] = 0

    for row in df_confrontos.itertuples():
        home, away = row.home_team, row.away_team
        home_score, away_score = row.home_score, row.away_score

        if home in qualified_teams: df_stats.at[home, "games_played"] += 1
        if away in qualified_teams: df_stats.at[away, "games_played"] += 1
        
        if home_score > away_score:
            if home in qualified_teams: df_stats.at[home, "wins"] += 1
            if away in qualified_teams: df_stats.at[away, "loses"] += 1
            
        elif away_score > home_score:
            if home in qualified_teams: df_stats.at[home, "loses"] += 1
            if away in qualified_teams: df_stats.at[away, "wins"] += 1
            
        else:
            if home in qualified_teams: df_stats.at[home, "draws"] += 1
            if away in qualified_teams: df_stats.at[away, "draws"] += 1

    df_stats["points"] = df_stats["wins"]*3 + df_stats["draws"]
    df_stats["points_per_game"] = round(df_stats["points"] / df_stats["games_played"],2)
    df_stats = df_stats.sort_values(by="points", ascending=False)
    df_stats.to_csv(f"dados/estatisticas_ciclos/estatisticas_ciclo{ano}.csv")

for ano in anos_copa:
    try:
        stats_cicle(ano)    
    except: ...


['Uruguay', 'Argentina', 'United States', 'Yugoslavia', 'Chile', 'Brazil', 'France', 'Romania', 'Paraguay', 'Peru', 'Belgium', 'Bolivia', 'Mexico']
['Italy', 'Czechoslovakia', 'Germany', 'Austria', 'Spain', 'Hungary', 'Switzerland', 'Sweden', 'Argentina', 'France', 'Netherlands', 'Romania', 'Egypt', 'Brazil', 'Belgium', 'United States']
['Italy', 'Hungary', 'Brazil', 'Sweden', 'Czechoslovakia', 'Switzerland', 'Cuba', 'France', 'Romania', 'Germany', 'Poland', 'Norway', 'Belgium', 'Netherlands', 'Dutch East Indies']
['Uruguay', 'Brazil', 'Sweden', 'Spain', 'Yugoslavia', 'Switzerland', 'Italy', 'England', 'Chile', 'United States', 'Paraguay', 'Mexico', 'Bolivia']
['West Germany', 'Hungary', 'Austria', 'Uruguay', 'Switzerland', 'Brazil', 'England', 'Yugoslavia', 'Turkey', 'Italy', 'France', 'Belgium', 'Mexico', 'Czechoslovakia', 'Scotland', 'South Korea']
['Brazil', 'Sweden', 'France', 'West Germany', 'Wales', 'Soviet Union', 'Northern Ireland', 'Yugoslavia', 'Czechoslovakia', 'Hungary', '

In [67]:
def cicle_games(df, ano_copa):
    #a função tá certa, mas para todos os dados ficarem certos tem que corrijir os csv e eu to com preguiça

    if ano_copa != 2022 and ano_copa != 2026:
        inicio_ciclo = date(year=ano_copa-4, month=8, day=1)
        fim_ciclo = date(year=ano_copa, month=5, day=26)
    
    elif ano_copa == 2022:
        inicio_ciclo = date(year=ano_copa-4, month=8, day=1)
        fim_ciclo = date(year=ano_copa, month=11, day=19)

    elif ano_copa == 2026:
        inicio_ciclo = date(year=ano_copa-4, month=12, day=20)
        fim_ciclo = date(year=ano_copa, month=6, day=10)

    selecoes = pd.read_csv(f'dados/copas/FIFA - {ano_copa}.csv')
    selecoes = selecoes['Team']
    jogos = df
    df_ciclo = jogos[(jogos['date'] >= inicio_ciclo) & (jogos['date'] <= fim_ciclo)]
    df_ciclo = df_ciclo[(df_ciclo['home_team'].isin(selecoes)) | (df_ciclo['away_team'].isin(selecoes))]
    df_ciclo = df_ciclo.reset_index()
    df_ciclo.to_csv(f"dados/confrontos/historico_confrontos_ciclo{ano_copa}.csv")
    stats_cicle(df, ano_copa)

for ano in anos_copa:
    try:
        cicle_games(df, ano)
    except: ...
